# CrossGate-Fusion (CGF-Net): Cross-Attentive, Channel-Gated CNN–Transformer Fusion for Aerial Scene Classification on AID

**Dataset:** AID (30 classes, 10,000 images) · **Split:** fixed stratified 70 / 10 / 20 · **Seeds:** 3 · **Backbones:** EfficientNet-B0 (CNN) and ViT-B/16 (Transformer), ImageNet-pretrained

---

## What this notebook does

| Stage | Purpose |
|---|---|
| 1. Setup, data audit | Deterministic split, exact-duplicate / leakage audit, dataset figures |
| 2. Models | Two single-backbone baselines, two simple fusion baselines, the proposed **CGF-Net**, and three ablations |
| 3. Training engine | One identical recipe for every model (fair comparison), crash-safe auto-resume |
| 4. Main results | Mean ± SD over seeds, late-fusion baselines, statistical tests, calibration |
| 5. Analysis | Learning curves, confusion structure, per-class F1, branch complementarity, gate behaviour, embedding geometry |
| 6. Stress tests | Robustness to noise, blur, low resolution, illumination, occlusion; label-efficiency; compute cost |
| 7. Export | Manuscript-ready tables (CSV / LaTeX) and 600-dpi, column-width figures |

## Problems found in the previous notebook and how they are fixed

| # | Issue in the earlier version | Consequence | Fix here |
|---|---|---|---|
| 1 | Fusion used **one scalar gate per image** on globally pooled vectors | Cannot express *which feature dimensions* to trust; a CNN/ViT trade-off collapses to one number | **Channel-wise gate** driven by `[c, v, abs(c−v), c⊙v]`, plus **bidirectional cross-attention** between CNN local tokens and ViT tokens |
| 2 | Single LR (2e-4) for a pretrained ViT and for new heads | Too aggressive for ViT fine-tuning; damages pretrained features | **Differential learning rates** (CNN / ViT / head), AdamW with no decay on bias/norm, warm-up + cosine, gradient clipping |
| 3 | Nothing forced each branch to stay discriminative | One branch can dominate ("modality laziness") | **Auxiliary branch heads** (deep supervision) and **branch dropout** |
| 4 | Augmentation: small rotation + h-flip only | Aerial scenes are rotation- and flip-invariant and scale-variable | RandomResizedCrop, h/v flips, right-angle rotations, random erasing; **dihedral TTA** at test time |
| 5 | Resume re-used the *same* shuffle order every resumed epoch | Non-equivalent to an uninterrupted run | Epoch-specific shuffle seed |
| 6 | Resuming a run that had **early-stopped** kept training | Violates the stopping rule | `stopped` flag stored in checkpoint |
| 7 | Reported LR was the *next* epoch's LR | Wrong learning-rate log | LR recorded before `scheduler.step()` |
| 8 | fp16 logits passed to `softmax` | Precision loss in probabilities / AUC (silent `NaN` via `try/except`) | Cast to float32, renormalise, no silent failure |
| 9 | `best_state` deep-copied on GPU **and** duplicated inside `latest.pth` | Doubled memory and disk | Best weights only in `best.pth`; atomic writes prevent corrupt checkpoints |
| 10 | Model selection by validation accuracy only | Ties resolved arbitrarily | Tie-break by validation loss |
| 11 | Running loss used `batch_size × n_batches` | Wrong on the last batch | Exact sample counts |
| 12 | Old checkpoints would be silently loaded after any hyper-parameter change | Invalid results | **Config fingerprint** guard per `RUN_ID` |
| 13 | Only accuracy-type metrics, no significance, no calibration, no ablations | Weak evidence for a publication | Full evaluation suite (Sections 4–6) |
| 14 | Figures saved at 300 dpi with `bbox_inches="tight"` | Unpredictable physical size, needs resizing in the manuscript | Figures are created **at final print size** (85 mm / 178 mm), 600 dpi PNG + vector PDF |

> **Honest scope note.** All numbers reported by this notebook are produced when you run it. Nothing is pre-filled. With a 70 % training split AID is close to saturation, so the notebook adds label-efficiency (20 % / 50 % training), robustness, and calibration experiments where the differences between models are more informative.

## Method summary — CGF-Net

Let $F_c\in\mathbb R^{49\times1280}$ be the last-stage EfficientNet-B0 feature map (flattened 7×7) and $F_v\in\mathbb R^{197\times768}$ the ViT-B/16 token sequence (CLS + 196 patches). Both are projected to a shared width $d=256$ with LayerNorm, giving $C$ and $V$.

**1. Bidirectional cross-attention.** Local CNN tokens query global ViT context, and the ViT CLS token queries the CNN local tokens:

$$\tilde C = \mathrm{LN}\big(C + \mathrm{MHA}(C, V, V)\big),\qquad \tilde v = \mathrm{LN}\big(V_{\text{cls}} + \mathrm{MHA}(V_{\text{cls}}, C, C)\big)$$

**2. Channel-wise gated fusion.** With $\bar c=\mathrm{mean}(\tilde C)$:

$$g=\sigma\!\big(W_2\,\mathrm{GELU}(W_1[\bar c;\tilde v;|\bar c-\tilde v|;\bar c\odot\tilde v])\big)\in(0,1)^{d},\qquad z=g\odot\bar c+(1-g)\odot\tilde v$$

**3. Training objective.** $\mathcal L=\mathrm{CE}(f(z),y)+\lambda\,[\mathrm{CE}(h_c(c_{raw}),y)+\mathrm{CE}(h_v(v_{raw}),y)]$ with auxiliary heads $h_c,h_v$ on the *pre-interaction* branch features, and with per-sample **branch dropout** ($p=0.1$) during training. Auxiliary heads are discarded at inference.

**Baselines and ablations (all share the same recipe):**

| Key | Description |
|---|---|
| `cnn` | EfficientNet-B0 + linear head |
| `vit` | ViT-B/16 + linear head |
| `concat` | Project both, concatenate, MLP (plain feature fusion) |
| `gate_scalar` | The previous notebook's design: one scalar gate per image, no attention, no aux loss |
| `cgf` | **Proposed** (cross-attention + channel gate + aux loss + branch dropout) |
| `cgf_noattn` | CGF without cross-attention |
| `cgf_nogate` | CGF with concatenation + MLP instead of the channel gate |
| `cgf_noaux` | CGF without auxiliary losses |
| `late_avg`, `late_val` | *Post-hoc* probability averaging of the trained `cnn` and `vit` models (equal / validation-tuned weight) — no extra training |

In [ ]:
# Cell 1 — Configuration and environment
from pathlib import Path
import os, sys, time, json, math, random, copy, gc, hashlib, warnings, platform
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import display

# ---------------------------------------------------------------------------
# Switch for a quick dry-run of the WHOLE pipeline on a tiny subset (minutes).
# Use it once to validate paths / GPU / plotting, then set False for the study.
# ---------------------------------------------------------------------------
SMOKE_TEST = False

# ----------------------------- paths ---------------------------------------
BASE_DIR     = Path(os.environ.get("AID_BASE_DIR",     r"C:\DSAI\SJ\AID STUDY CONFERENCE"))
DATASET_ROOT = Path(os.environ.get("AID_DATASET_ROOT", r"D:\SJ\UVMDATA\AID\AID"))

# NEW run id: the architecture and recipe changed, so old checkpoints must NOT be reused.
RUN_ID = "AID_CGF_01"

# ----------------------------- experiment plan -----------------------------
SEEDS           = [42, 123, 2026]
MAIN_MODELS     = ["cnn", "vit", "concat", "gate_scalar", "cgf"]
ABLATION_MODELS = ["cgf_noattn", "cgf_nogate", "cgf_noaux"]
RUN_ABLATIONS   = True

RUN_LOW_DATA        = True             # label-efficiency study (nested stratified subsets of the train split)
LOW_DATA_FRACTIONS  = [0.2, 0.5]
LOW_DATA_MODELS     = ["cnn", "vit", "cgf"]
LOW_DATA_SEEDS      = [42]

RUN_ROBUSTNESS      = True
CHECK_DUPLICATES    = True             # exact-duplicate (MD5) leakage audit

# ----------------------------- training recipe (identical for ALL models) --
IMAGE_SIZE        = 224
BATCH_SIZE        = 8                  # per step; lower if you hit CUDA OOM
GRAD_ACCUM_STEPS  = 2                  # effective batch = 16
EVAL_BATCH_SIZE   = 32
NUM_WORKERS       = 2                  # if the DataLoader check in Cell 6 hangs, set this to 0 (slower but always works)

EPOCHS            = 25
WARMUP_EPOCHS     = 2
MIN_EPOCHS        = 10                 # early stopping is not allowed before this epoch
PATIENCE          = 8

LR_CNN            = 1e-4               # pretrained CNN backbone
LR_VIT            = 3e-5               # pretrained ViT backbone (must be small)
LR_HEAD           = 5e-4               # newly initialised layers
MIN_LR_RATIO      = 0.01
WEIGHT_DECAY      = 0.05
LABEL_SMOOTHING   = 0.10
GRAD_CLIP         = 1.0

# ----------------------------- CGF-Net -------------------------------------
FUSION_DIM        = 256
FUSION_HEADS      = 4
AUX_WEIGHT        = 0.30
BRANCH_DROP       = 0.10

# ----------------------------- split ---------------------------------------
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.10, 0.20
SPLIT_SEED        = 2026
SMOKE_IMAGES_PER_CLASS = 14

# ----------------------------- housekeeping --------------------------------
PRETRAINED                 = True
DELETE_LATEST_WHEN_DONE    = True      # frees ~1 GB per finished run; resume logic does not need it once metrics exist
KEEP_BEST_FOR_ANALYSIS     = True      # keep best.pth of SEEDS[0] (full data) for gate / t-SNE / robustness analyses

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

if SMOKE_TEST:
    RUN_ID += "_SMOKE"
    EPOCHS, WARMUP_EPOCHS, MIN_EPOCHS, PATIENCE = 3, 1, 3, 3
    SEEDS, LOW_DATA_SEEDS, LOW_DATA_FRACTIONS = [42], [42], [0.5]
    NUM_WORKERS = 0

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# ----------------------------- directories ---------------------------------
RUN_ROOT        = BASE_DIR / "RESULTS" / "FRESH_EXPERIMENT" / RUN_ID
SPLIT_ROOT      = RUN_ROOT / "SPLITS"
CHECKPOINT_ROOT = RUN_ROOT / "CHECKPOINTS"
MODEL_ROOT      = RUN_ROOT / "MODELS"          # per-run metric JSON files
HISTORY_ROOT    = RUN_ROOT / "HISTORY"
PRED_ROOT       = RUN_ROOT / "PREDICTIONS"
FIG_ROOT        = RUN_ROOT / "FIGURES"
TABLE_ROOT      = RUN_ROOT / "TABLES"
for p in [RUN_ROOT, SPLIT_ROOT, CHECKPOINT_ROOT, MODEL_ROOT, HISTORY_ROOT, PRED_ROOT, FIG_ROOT, TABLE_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

# ----------------------------- config fingerprint guard --------------------
CFG = dict(image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, accum=GRAD_ACCUM_STEPS, epochs=EPOCHS,
           warmup=WARMUP_EPOCHS, min_epochs=MIN_EPOCHS, patience=PATIENCE, lr_cnn=LR_CNN, lr_vit=LR_VIT,
           lr_head=LR_HEAD, min_lr_ratio=MIN_LR_RATIO, wd=WEIGHT_DECAY, ls=LABEL_SMOOTHING, clip=GRAD_CLIP,
           fusion_dim=FUSION_DIM, fusion_heads=FUSION_HEADS, aux_w=AUX_WEIGHT, branch_drop=BRANCH_DROP,
           split=(TRAIN_RATIO, VAL_RATIO, TEST_RATIO), split_seed=SPLIT_SEED, pretrained=PRETRAINED,
           smoke=SMOKE_TEST)
FINGERPRINT = hashlib.md5(json.dumps(CFG, sort_keys=True, default=str).encode()).hexdigest()[:10]
cfg_path = RUN_ROOT / "config.json"
if cfg_path.exists():
    old = json.loads(cfg_path.read_text(encoding="utf-8"))
    if old.get("fingerprint") != FINGERPRINT:
        raise RuntimeError(
            f"Hyper-parameters differ from the ones used when RUN_ID='{RUN_ID}' was created "
            f"(old {old.get('fingerprint')} vs new {FINGERPRINT}).\n"
            "Resuming with different settings would silently invalidate the study.\n"
            "-> Restore the original settings, or use a new RUN_ID.")
else:
    cfg_path.write_text(json.dumps(dict(fingerprint=FINGERPRINT, config=CFG,
        torch=torch.__version__, python=sys.version.split()[0], platform=platform.platform()),
        indent=2, default=str), encoding="utf-8")

print("=" * 90)
print(f"RUN_ID {RUN_ID} | fingerprint {FINGERPRINT} | smoke={SMOKE_TEST}")
print("=" * 90)
print(f"Python {sys.version.split()[0]} | PyTorch {torch.__version__} | device {DEVICE} | AMP {USE_AMP}")
if torch.cuda.is_available():
    print(f"GPU {torch.cuda.get_device_name(0)} | {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")
print(f"Dataset : {DATASET_ROOT}\nRun dir : {RUN_ROOT}")
assert DATASET_ROOT.exists(), f"Dataset folder not found: {DATASET_ROOT}. Edit DATASET_ROOT above."
print("✓ Configuration ready.")

In [ ]:
# Cell 2 — Utilities: seeding, run tags, publication figure style, figure export

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def run_tag(model, seed, frac=1.0):
    return f"{model}_seed{seed}" if frac >= 1.0 else f"{model}_tr{int(round(frac*100))}_seed{seed}"

# ---- Publication figure style (figures are drawn at FINAL print size) ------
FIG_DPI = 600
COL1    = 3.35        # single column  ≈ 85 mm
COL2    = 7.00        # double column  ≈ 178 mm
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
    "font.size": 7, "axes.titlesize": 7.5, "axes.labelsize": 7,
    "xtick.labelsize": 6.5, "ytick.labelsize": 6.5, "legend.fontsize": 6.3,
    "axes.linewidth": 0.6, "lines.linewidth": 1.0, "lines.markersize": 3.0,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6, "xtick.major.size": 2.5, "ytick.major.size": 2.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.4,
    "legend.frameon": False, "figure.dpi": 130, "savefig.dpi": FIG_DPI,
    "pdf.fonttype": 42, "ps.fonttype": 42, "axes.titlepad": 3,
})

# Colour-blind-safe (Okabe–Ito) palette, one fixed colour per model across ALL figures.
MODEL_STYLE = {
    "cnn":         ("EfficientNet-B0",        "#0072B2", "o"),
    "vit":         ("ViT-B/16",               "#D55E00", "s"),
    "late_avg":    ("Late fusion (avg)",      "#7F7F7F", "v"),
    "late_val":    ("Late fusion (val-tuned)","#4D4D4D", "^"),
    "concat":      ("Concat fusion",          "#CC79A7", "D"),
    "gate_scalar": ("Scalar-gate fusion",     "#E69F00", "P"),
    "cgf":         ("CGF-Net (proposed)",     "#009E73", "*"),
    "cgf_noattn":  ("CGF w/o cross-attn",     "#56B4E9", "X"),
    "cgf_nogate":  ("CGF w/o channel gate",   "#F0A3A3", "h"),
    "cgf_noaux":   ("CGF w/o aux loss",       "#000000", "p"),
}
LABEL = {k: v[0] for k, v in MODEL_STYLE.items()}
COLOR = {k: v[1] for k, v in MODEL_STYLE.items()}
MARKER = {k: v[2] for k, v in MODEL_STYLE.items()}

FIG_LOG = []
def save_fig(fig, name, show=True):
    # No bbox_inches="tight": the file keeps the exact physical size it was designed at.
    w, h = fig.get_size_inches()
    png = FIG_ROOT / f"{name}.png"
    fig.savefig(png, dpi=FIG_DPI)
    fig.savefig(FIG_ROOT / f"{name}.pdf")
    with Image.open(png) as im:
        px = im.size
        im.save(png, optimize=True, dpi=(FIG_DPI, FIG_DPI))
    kb = png.stat().st_size / 1024
    FIG_LOG.append(dict(figure=name, width_in=round(w, 2), height_in=round(h, 2),
                        width_mm=round(w*25.4, 1), pixels=f"{px[0]}x{px[1]}", dpi=FIG_DPI, png_kb=round(kb, 1)))
    print(f"saved {name}.png  {w:.2f}x{h:.2f} in  ({px[0]}x{px[1]} px @ {FIG_DPI} dpi, {kb:.0f} KB) + PDF")
    if show: plt.show()
    plt.close(fig)

print("✓ Utilities and figure style ready.")

## 1 · Data: discovery, fixed split, integrity audit

In [ ]:
# Cell 3 — Discover AID images; create (or reuse) the fixed stratified 70/10/20 split
from sklearn.model_selection import train_test_split

SPLIT_TRAIN, SPLIT_VAL = SPLIT_ROOT / "train_index.csv", SPLIT_ROOT / "val_index.csv"
SPLIT_TEST,  SPLIT_ALL = SPLIT_ROOT / "test_index.csv",  SPLIT_ROOT / "dataset_index.csv"

if all(p.exists() for p in [SPLIT_TRAIN, SPLIT_VAL, SPLIT_TEST, SPLIT_ALL]):
    print("Reusing the split saved for this RUN_ID.")
    dataset_df, train_df = pd.read_csv(SPLIT_ALL), pd.read_csv(SPLIT_TRAIN)
    val_df, test_df = pd.read_csv(SPLIT_VAL), pd.read_csv(SPLIT_TEST)
else:
    print("Creating dataset index and fixed split ...")
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
    records = []
    class_dirs = sorted(p for p in DATASET_ROOT.iterdir() if p.is_dir())
    for cdir in tqdm(class_dirs, desc="Scanning classes"):
        files = sorted(f for f in cdir.rglob("*") if f.is_file() and f.suffix.lower() in exts)
        if SMOKE_TEST:
            files = files[:SMOKE_IMAGES_PER_CLASS]
        records += [{"path": str(f.resolve()), "class": cdir.name} for f in files]
    dataset_df = pd.DataFrame(records)
    if len(dataset_df) == 0:
        raise RuntimeError(f"No images found under {DATASET_ROOT}")
    cls_list = sorted(dataset_df["class"].unique())
    if len(cls_list) != 30:
        raise RuntimeError(f"Expected 30 AID classes, found {len(cls_list)}")
    dataset_df["label"] = dataset_df["class"].map({c: i for i, c in enumerate(cls_list)})

    trval_df, test_df = train_test_split(dataset_df, test_size=TEST_RATIO,
                                         stratify=dataset_df["label"], random_state=SPLIT_SEED)
    train_df, val_df = train_test_split(trval_df, test_size=VAL_RATIO / (TRAIN_RATIO + VAL_RATIO),
                                        stratify=trval_df["label"], random_state=SPLIT_SEED)
    key = ["label", "path"]
    train_df, val_df, test_df, dataset_df = [d.sort_values(key).reset_index(drop=True)
                                             for d in (train_df, val_df, test_df, dataset_df)]
    dataset_df.to_csv(SPLIT_ALL, index=False); train_df.to_csv(SPLIT_TRAIN, index=False)
    val_df.to_csv(SPLIT_VAL, index=False);     test_df.to_csv(SPLIT_TEST, index=False)
    print("✓ Split saved inside this experiment folder.")

classes      = sorted(dataset_df["class"].unique())
NUM_CLASSES  = len(classes)
assert NUM_CLASSES == 30
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    missing = [p for p in d["path"] if not Path(p).exists()]
    assert not missing, f"{len(missing)} missing files in {name}, e.g. {missing[0]}"
assert not (set(train_df.path) & set(test_df.path)) and not (set(train_df.path) & set(val_df.path)) \
       and not (set(val_df.path) & set(test_df.path)), "Path overlap between splits!"

summary = pd.DataFrame({"images": [len(dataset_df), len(train_df), len(val_df), len(test_df)]},
                       index=["Total", "Train", "Validation", "Test"])
summary["share (%)"] = (100 * summary["images"] / len(dataset_df)).round(1)
summary.to_csv(TABLE_ROOT / "table_dataset_split.csv")
display(summary)
print(f"Classes: {NUM_CLASSES} | min/max images per class: "
      f"{dataset_df['class'].value_counts().min()} / {dataset_df['class'].value_counts().max()}")
print("✓ All files exist, no path overlap between splits.")

In [ ]:
# Cell 4 — Leakage audit: exact-duplicate detection across splits (MD5)
def file_md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

if CHECK_DUPLICATES:
    hash_csv = SPLIT_ROOT / "file_md5.csv"
    cache = pd.read_csv(hash_csv).set_index("path")["md5"].to_dict() if hash_csv.exists() else {}
    todo = [p for p in dataset_df["path"] if p not in cache]
    for p in tqdm(todo, desc="Hashing files", disable=(len(todo) == 0)):
        cache[p] = file_md5(p)
    pd.Series(cache, name="md5").rename_axis("path").reset_index().to_csv(hash_csv, index=False)

    tr, va, te = (set(cache[p] for p in d["path"]) for d in (train_df, val_df, test_df))
    n_dup_all = int(dataset_df["path"].map(cache).duplicated().sum())
    audit = pd.DataFrame({"count": [n_dup_all, len(tr & te), len(tr & va), len(va & te)]},
                         index=["duplicate files in dataset", "train∩test identical images",
                                "train∩val identical images", "val∩test identical images"])
    audit.to_csv(TABLE_ROOT / "table_leakage_audit.csv")
    display(audit)
    if audit["count"].iloc[1:].sum() > 0:
        print("⚠ Identical images cross splits — report this in the paper or de-duplicate before training.")
    else:
        print("✓ No byte-identical image appears in more than one split.")
else:
    print("Duplicate audit skipped (CHECK_DUPLICATES=False).")

In [ ]:
# Cell 5 — Dataset figures (class balance + sample gallery), sized for the manuscript
counts = dataset_df["class"].value_counts().reindex(classes)
fig, ax = plt.subplots(figsize=(COL2, 2.0), layout="constrained")
ax.bar(range(NUM_CLASSES), counts.values, color="#0072B2", width=0.75)
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(classes, rotation=90, fontsize=5.5)
ax.set_ylabel("Images"); ax.grid(axis="x", visible=False); ax.set_xlim(-0.7, NUM_CLASSES - 0.3)
save_fig(fig, "fig01_class_distribution")

rng = np.random.RandomState(0)
n_r, n_c = 5, 6
fig, axes = plt.subplots(n_r, n_c, figsize=(COL2, COL2 * n_r / n_c * 0.93), layout="constrained")
for i, (ax, cname) in enumerate(zip(axes.ravel(), classes)):
    p = dataset_df[dataset_df["class"] == cname]["path"].iloc[rng.randint((dataset_df["class"] == cname).sum())]
    with Image.open(p) as im:
        ax.imshow(im.convert("RGB").resize((160, 160)))
    ax.set_title(cname, fontsize=6, pad=1.5); ax.axis("off")
save_fig(fig, "fig02_sample_gallery")

In [ ]:
# Cell 6 — Datasets, aerial-specific augmentation, DataLoaders
# Worker-safe data code: on Windows, DataLoader workers are *spawned* and must import these
# classes/functions. Anything defined inside a notebook cannot be imported, which makes workers hang.
# So they live in a real module file that is written next to the results and imported.
AID_DATA_MODULE = r"""
import random
import numpy as np
import torch
from torch.utils.data import Dataset
from PIL import Image

class AIDDataset(Dataset):
    def __init__(self, df, transform=None):
        self.paths = df["path"].tolist(); self.labels = df["label"].astype(int).tolist(); self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, self.labels[i]

class RandomRightAngleRotation:
    # Aerial scenes have no canonical up: rotate by a random multiple of 90 degrees.
    def __call__(self, img):
        k = random.choice([0, 90, 180, 270])
        return img.rotate(k) if k else img

def seed_worker(worker_id):
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)
"""
CODE_DIR = RUN_ROOT / "CODE"; CODE_DIR.mkdir(exist_ok=True)
(CODE_DIR / "aid_data.py").write_text(AID_DATA_MODULE, encoding="utf-8")
if str(CODE_DIR) not in sys.path: sys.path.insert(0, str(CODE_DIR))
import importlib, aid_data; importlib.reload(aid_data)
from aid_data import AIDDataset, RandomRightAngleRotation, seed_worker

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.45, 1.0), ratio=(0.85, 1.18),
                                 interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.RandomHorizontalFlip(0.5), transforms.RandomVerticalFlip(0.5),
    RandomRightAngleRotation(),
    transforms.ColorJitter(0.20, 0.20, 0.15, 0.02),
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.20, scale=(0.02, 0.12), value="random"),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def subsample_train(frac):
    """Nested, class-stratified subsets of the training split (frac=0.2 ⊂ frac=0.5 ⊂ 1.0)."""
    if frac >= 1.0: return train_df
    order = np.random.RandomState(SPLIT_SEED).permutation(len(train_df))
    shuffled = train_df.iloc[order]
    parts = [g.head(int(math.ceil(len(g) * frac))) for _, g in shuffled.groupby("label")]
    return pd.concat(parts).sort_values(["label", "path"]).reset_index(drop=True)

def make_loaders(seed, frac=1.0):
    g = torch.Generator(); g.manual_seed(seed)
    tr_df = subsample_train(frac)
    common = dict(num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
                  persistent_workers=(NUM_WORKERS > 0), worker_init_fn=seed_worker)
    train_loader = DataLoader(AIDDataset(tr_df, train_transform), batch_size=BATCH_SIZE, shuffle=True,
                              drop_last=True, generator=g, **common)
    val_loader  = DataLoader(AIDDataset(val_df,  eval_transform), batch_size=EVAL_BATCH_SIZE, shuffle=False, **common)
    test_loader = DataLoader(AIDDataset(test_df, eval_transform), batch_size=EVAL_BATCH_SIZE, shuffle=False, **common)
    return train_loader, val_loader, test_loader, len(tr_df)

_tl, _vl, _sl, _n = make_loaders(SEEDS[0])
_t0 = time.time(); _xb, _yb = next(iter(_tl))
print(f"✓ DataLoader check: first batch {tuple(_xb.shape)} in {time.time()-_t0:.1f}s with NUM_WORKERS={NUM_WORKERS}")
del _xb, _yb
print(f"Train {_n} imgs / {len(_tl)} steps | Val {len(val_df)} imgs | Test {len(test_df)} imgs")
for fr in LOW_DATA_FRACTIONS:
    print(f"  low-data subset {fr:.0%}: {len(subsample_train(fr))} training images")
del _tl, _vl, _sl

## 2 · Models

In [ ]:
# Cell 7 — Models: CNN, ViT, fusion baselines, CGF-Net and ablations

def cnn_backbone(pretrained=None):
    pretrained = PRETRAINED if pretrained is None else pretrained
    w = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
    return models.efficientnet_b0(weights=w).features           # (B,1280,7,7) at 224px

def vit_backbone(pretrained=None):
    pretrained = PRETRAINED if pretrained is None else pretrained
    w = models.ViT_B_16_Weights.IMAGENET1K_V1 if pretrained else None
    m = models.vit_b_16(weights=w)
    m.heads = nn.Identity()
    return m

def vit_tokens(vit, x):
    """Full token sequence (B,197,768) after the final encoder LayerNorm."""
    x = vit._process_input(x)
    x = torch.cat([vit.class_token.expand(x.shape[0], -1, -1), x], dim=1)
    return vit.encoder(x)


class CNNClassifier(nn.Module):
    def __init__(self, num_classes, pretrained=None):
        super().__init__()
        self.cnn = cnn_backbone(pretrained)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(1280, num_classes))
    def embed(self, x): return self.pool(self.cnn(x)).flatten(1)
    def forward(self, x): return self.head(self.embed(x))


class ViTClassifier(nn.Module):
    def __init__(self, num_classes, pretrained=None):
        super().__init__()
        self.vit = vit_backbone(pretrained)
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(768, num_classes))
    def embed(self, x): return self.vit(x)                       # CLS token
    def forward(self, x): return self.head(self.embed(x))


class FusionNet(nn.Module):
    """
    kind      : 'concat' | 'scalar_gate' | 'channel_gate'
    xattn     : bidirectional cross-attention between CNN tokens and ViT tokens
    aux       : auxiliary classification heads on pre-interaction branch features (training only)
    branch_drop: probability of blanking one branch per sample (training only)
    """
    def __init__(self, num_classes, kind, xattn, aux, branch_drop, dim=None, heads=None, pretrained=None):
        super().__init__()
        dim = dim or FUSION_DIM; heads = heads or FUSION_HEADS
        self.kind, self.xattn, self.aux, self.branch_drop = kind, xattn, aux, branch_drop
        self.cnn = cnn_backbone(pretrained)
        self.vit = vit_backbone(pretrained)

        self.cnn_tok = nn.Sequential(nn.Linear(1280, dim), nn.LayerNorm(dim))
        self.vit_tok = nn.Sequential(nn.Linear(768,  dim), nn.LayerNorm(dim))
        self.drop = nn.Dropout(0.10)

        if xattn:
            self.c2v = nn.MultiheadAttention(dim, heads, dropout=0.10, batch_first=True)  # local CNN <- global ViT
            self.v2c = nn.MultiheadAttention(dim, heads, dropout=0.10, batch_first=True)  # ViT CLS  <- local CNN
            self.norm_c, self.norm_v = nn.LayerNorm(dim), nn.LayerNorm(dim)

        if kind == "channel_gate":
            self.gate = nn.Sequential(nn.Linear(4 * dim, dim), nn.GELU(), nn.Dropout(0.10), nn.Linear(dim, dim))
        elif kind == "scalar_gate":
            self.gate = nn.Sequential(nn.Linear(2 * dim, dim), nn.GELU(), nn.Dropout(0.10), nn.Linear(dim, 1))
        elif kind == "concat":
            self.merge = nn.Sequential(nn.Linear(2 * dim, dim), nn.LayerNorm(dim), nn.GELU())
        else:
            raise ValueError(kind)

        self.classifier = nn.Sequential(nn.LayerNorm(dim), nn.Dropout(0.20), nn.Linear(dim, dim), nn.GELU(),
                                        nn.Dropout(0.20), nn.Linear(dim, num_classes))
        if aux:
            self.aux_c = nn.Linear(dim, num_classes)
            self.aux_v = nn.Linear(dim, num_classes)

    def encode(self, x):
        ct = self.cnn_tok(self.cnn(x).flatten(2).transpose(1, 2))      # (B,49,D)
        vt = self.vit_tok(vit_tokens(self.vit, x))                      # (B,197,D)
        v_cls = vt[:, :1]                                               # (B,1,D)
        c_raw, v_raw = ct.mean(1), v_cls.squeeze(1)
        if self.xattn:
            c_ctx, _ = self.c2v(ct, vt, vt, need_weights=False)
            v_ctx, _ = self.v2c(v_cls, ct, ct, need_weights=False)
            c_vec = self.norm_c(ct + c_ctx).mean(1)
            v_vec = self.norm_v(v_cls + v_ctx).squeeze(1)
        else:
            c_vec, v_vec = c_raw, v_raw
        return c_vec, v_vec, c_raw, v_raw

    def fuse(self, c, v):
        if self.kind == "channel_gate":
            g = torch.sigmoid(self.gate(torch.cat([c, v, (c - v).abs(), c * v], 1)))
            return g * c + (1 - g) * v, g
        if self.kind == "scalar_gate":
            g = torch.sigmoid(self.gate(torch.cat([c, v], 1)))
            return g * c + (1 - g) * v, g
        return self.merge(torch.cat([c, v], 1)), None

    def forward(self, x, return_all=False):
        c, v, c_raw, v_raw = self.encode(x)
        c, v = self.drop(c), self.drop(v)
        if self.training and self.branch_drop > 0:
            r = torch.rand(c.shape[0], 1, device=c.device)
            c = c * (~(r < self.branch_drop / 2)).to(c.dtype)
            v = v * (~((r >= self.branch_drop / 2) & (r < self.branch_drop))).to(v.dtype)
        fused, g = self.fuse(c, v)
        logits = self.classifier(fused)
        if return_all:
            out = dict(logits=logits, gate=g, feat=fused)
            if self.aux:
                out["cnn_logits"], out["vit_logits"] = self.aux_c(c_raw), self.aux_v(v_raw)
            return out
        if self.training and self.aux:
            return logits, self.aux_c(c_raw), self.aux_v(v_raw)
        return logits

    def embed(self, x): return self.forward(x, return_all=True)["feat"]


FUSION_CONFIGS = {
    "concat":      dict(kind="concat",       xattn=False, aux=False, branch_drop=0.0),
    "gate_scalar": dict(kind="scalar_gate",  xattn=False, aux=False, branch_drop=0.0),   # previous notebook's design
    "cgf":         dict(kind="channel_gate", xattn=True,  aux=True,  branch_drop=BRANCH_DROP),
    "cgf_noattn":  dict(kind="channel_gate", xattn=False, aux=True,  branch_drop=BRANCH_DROP),
    "cgf_nogate":  dict(kind="concat",       xattn=True,  aux=True,  branch_drop=BRANCH_DROP),
    "cgf_noaux":   dict(kind="channel_gate", xattn=True,  aux=False, branch_drop=BRANCH_DROP),
}

def build_model(name, pretrained=None):
    if name == "cnn": return CNNClassifier(NUM_CLASSES, pretrained)
    if name == "vit": return ViTClassifier(NUM_CLASSES, pretrained)
    if name in FUSION_CONFIGS: return FusionNet(NUM_CLASSES, pretrained=pretrained, **FUSION_CONFIGS[name])
    raise ValueError(name)

def count_params(m): return sum(p.numel() for p in m.parameters())

def build_optimizer(model):
    """Differential LR (CNN / ViT / new layers); no weight decay on biases, norms, tokens, pos-embeddings."""
    groups = {}
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        lr = LR_CNN if n.startswith("cnn.") else LR_VIT if n.startswith("vit.") else LR_HEAD
        no_wd = p.ndim <= 1 or n.endswith(".bias") or "class_token" in n or "pos_embedding" in n
        groups.setdefault((lr, 0.0 if no_wd else WEIGHT_DECAY), []).append(p)
    return torch.optim.AdamW([dict(params=ps, lr=lr, weight_decay=wd) for (lr, wd), ps in groups.items()])

def lr_lambda(epoch):
    """Linear warm-up followed by cosine decay (epoch-wise; resume-safe)."""
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / (WARMUP_EPOCHS + 1)
    prog = (epoch - WARMUP_EPOCHS) / max(1, EPOCHS - WARMUP_EPOCHS)
    return MIN_LR_RATIO + (1 - MIN_LR_RATIO) * 0.5 * (1 + math.cos(math.pi * min(1.0, prog)))

# Quick shape / parameter check with random weights (no download needed)
_x = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE)
rows = []
for name in MAIN_MODELS + ABLATION_MODELS:
    m = build_model(name, pretrained=False).eval()
    with torch.no_grad(): out = m(_x)
    assert out.shape == (2, NUM_CLASSES), (name, out.shape)
    rows.append(dict(model=LABEL[name], params_M=round(count_params(m) / 1e6, 2)))
    del m
display(pd.DataFrame(rows)); del _x
print("✓ All architectures build and produce (B, 30) logits.")

## 3 · Training engine (one recipe for all models · crash-safe · auto-resume)

In [ ]:
# Cell 8 — Metrics and evaluation
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score,
                             matthews_corrcoef, cohen_kappa_score, roc_auc_score, classification_report,
                             confusion_matrix)

def expected_calibration_error(probs, y, n_bins=15):
    conf, pred = probs.max(1), probs.argmax(1)
    acc = (pred == y).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): ece += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(ece)

def compute_metrics(y_true, probs):
    probs = np.asarray(probs, dtype=np.float64)
    probs = probs / probs.sum(1, keepdims=True)
    y_pred = probs.argmax(1)
    onehot = np.eye(probs.shape[1])[y_true]
    m = dict(
        accuracy=accuracy_score(y_true, y_pred),
        balanced_accuracy=balanced_accuracy_score(y_true, y_pred),
        macro_precision=precision_score(y_true, y_pred, average="macro", zero_division=0),
        macro_recall=recall_score(y_true, y_pred, average="macro", zero_division=0),
        macro_f1=f1_score(y_true, y_pred, average="macro", zero_division=0),
        weighted_f1=f1_score(y_true, y_pred, average="weighted", zero_division=0),
        mcc=matthews_corrcoef(y_true, y_pred), cohen_kappa=cohen_kappa_score(y_true, y_pred),
        nll=float(-np.log(np.clip(probs[np.arange(len(y_true)), y_true], 1e-12, 1)).mean()),
        brier=float(((probs - onehot) ** 2).sum(1).mean()),
        ece=expected_calibration_error(probs, y_true),
    )
    try:
        m["roc_auc_ovr_macro"] = roc_auc_score(y_true, probs, multi_class="ovr", average="macro",
                                               labels=list(range(probs.shape[1])))
    except ValueError as e:                     # e.g. a class absent in a tiny smoke-test split — be explicit
        print("ROC-AUC not computable:", e); m["roc_auc_ovr_macro"] = float("nan")
    return {k: float(v) for k, v in m.items()}

DIHEDRAL = [(k, f) for k in range(4) for f in (False, True)]       # the 8 symmetries of the square
def dihedral(x, k, flip):
    x = torch.rot90(x, k, (2, 3))
    return torch.flip(x, (3,)) if flip else x

@torch.no_grad()
def predict_probs(model, loader, tta=False, desc="EVAL", transform_fn=None):
    """Returns (y_true, probs[float64]). transform_fn lets robustness tests perturb un-normalised batches."""
    model.eval()
    ys, ps = [], []
    for images, labels in tqdm(loader, desc=desc, leave=False, dynamic_ncols=True):
        images = images.to(DEVICE, non_blocking=True)
        if transform_fn is not None: images = transform_fn(images)
        p = 0
        for k, f in (DIHEDRAL if tta else [(0, False)]):
            with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
                logits = model(dihedral(images, k, f))
            p = p + torch.softmax(logits.float(), dim=1)
        ps.append((p / (len(DIHEDRAL) if tta else 1)).cpu().numpy()); ys.append(labels.numpy())
    return np.concatenate(ys), np.concatenate(ps).astype(np.float64)

print("✓ Metrics and evaluation ready.")

In [ ]:
# Cell 9 — Paths, atomic checkpoint writes, GPU-memory helper
def get_paths(tag):
    return {
        "dir":         CHECKPOINT_ROOT / tag,
        "latest":      CHECKPOINT_ROOT / tag / "latest.pth",
        "best":        CHECKPOINT_ROOT / tag / "best.pth",
        "history":     HISTORY_ROOT / f"{tag}_history.csv",
        "metrics":     MODEL_ROOT / f"{tag}_metrics.json",
        "predictions": PRED_ROOT / f"{tag}_test_predictions.npz",
    }

def atomic_save(obj, path):
    """Write to a temp file, then rename: a crash mid-write can never corrupt an existing checkpoint."""
    tmp = Path(str(path) + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

def gpu_memory():
    if not torch.cuda.is_available(): return "CPU"
    return f"{torch.cuda.max_memory_allocated()/1024**3:.1f} GB peak"

print("✓ Checkpoint utilities ready.")

In [ ]:
# Cell 10 — Training / validation loops
def train_one_epoch(model, loader, criterion, optimizer, scaler, epoch):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    n_seen, loss_sum, correct = 0, 0.0, 0
    steps = len(loader)
    pbar = tqdm(loader, desc=f"  train {epoch:02d}/{EPOCHS}", leave=False, dynamic_ncols=True)
    for i, (images, labels) in enumerate(pbar, 1):
        images = images.to(DEVICE, non_blocking=True); labels = labels.to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
            out = model(images)
            if isinstance(out, tuple):                              # fusion model with auxiliary heads
                logits, aux_c, aux_v = out
                loss = criterion(logits, labels) + AUX_WEIGHT * (criterion(aux_c, labels) + criterion(aux_v, labels))
            else:
                logits = out; loss = criterion(logits, labels)
        scaler.scale(loss / GRAD_ACCUM_STEPS).backward()
        if i % GRAD_ACCUM_STEPS == 0 or i == steps:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(set_to_none=True)
        with torch.no_grad():
            bs = labels.size(0); n_seen += bs
            loss_sum += F.cross_entropy(logits.float(), labels).item() * bs      # un-smoothed CE, comparable to validation
            correct += (logits.argmax(1) == labels).sum().item()
        if i % 20 == 0 or i == steps:
            pbar.set_postfix(loss=f"{loss_sum/n_seen:.4f}", acc=f"{correct/n_seen:.4f}")
    return loss_sum / n_seen, correct / n_seen

@torch.no_grad()
def validate_one_epoch(model, loader, epoch):
    model.eval()
    n_seen, loss_sum, correct = 0, 0.0, 0
    for images, labels in tqdm(loader, desc=f"  valid {epoch:02d}/{EPOCHS}", leave=False, dynamic_ncols=True):
        images = images.to(DEVICE, non_blocking=True); labels = labels.to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
            logits = model(images)
        bs = labels.size(0); n_seen += bs
        loss_sum += F.cross_entropy(logits.float(), labels).item() * bs
        correct += (logits.argmax(1) == labels).sum().item()
    return loss_sum / n_seen, correct / n_seen

print("✓ Training loops ready.")

In [ ]:
# Cell 11 — Train / auto-resume one (model, seed, train-fraction) run
def run_single_experiment(model_name, seed, frac=1.0):
    tag = run_tag(model_name, seed, frac)
    paths = get_paths(tag); paths["dir"].mkdir(parents=True, exist_ok=True)
    seed_everything(seed)
    torch.cuda.reset_peak_memory_stats() if torch.cuda.is_available() else None

    print("\n" + "=" * 100 + f"\n{tag}  |  {LABEL.get(model_name, model_name)}  |  train fraction {frac:.0%}\n" + "=" * 100)
    model = build_model(model_name).to(DEVICE)
    n_params = count_params(model)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer = build_optimizer(model)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    train_loader, val_loader, test_loader, n_train = make_loaders(seed, frac)

    meta = dict(epoch=0, best_val_acc=-1.0, best_val_loss=float("inf"), best_epoch=0,
                history=[], patience=0, stopped=False)

    if paths["latest"].exists():                                     # ---- automatic resume
        ck = torch.load(paths["latest"], map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model_state"]); optimizer.load_state_dict(ck["optimizer_state"])
        scheduler.load_state_dict(ck["scheduler_state"]); scaler.load_state_dict(ck["scaler_state"])
        meta.update(ck["meta"]); del ck
        print(f"↻ RESUME: completed epoch {meta['epoch']} | best val acc {meta['best_val_acc']:.4f} "
              f"(epoch {meta['best_epoch']}) | early-stopped={meta['stopped']}")
    else:
        print("● Fresh start from epoch 1")

    for epoch in range(meta["epoch"] + 1, EPOCHS + 1):
        if meta["stopped"]: break
        t0 = time.time()
        train_loader.generator.manual_seed(seed * 1000 + epoch)      # epoch-specific shuffle => resume ≡ uninterrupted
        lr_now = lr_lambda(epoch - 1)                                # LR multiplier applied during this epoch
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler, epoch)
        va_loss, va_acc = validate_one_epoch(model, val_loader, epoch)
        scheduler.step()

        improved = (va_acc > meta["best_val_acc"] + 1e-12) or \
                   (abs(va_acc - meta["best_val_acc"]) <= 1e-12 and va_loss < meta["best_val_loss"])
        if improved:
            meta.update(best_val_acc=va_acc, best_val_loss=va_loss, best_epoch=epoch, patience=0)
            atomic_save({"epoch": epoch, "val_accuracy": va_acc, "model_state": model.state_dict()}, paths["best"])
        else:
            meta["patience"] += 1
        meta["history"].append(dict(epoch=epoch, train_loss=tr_loss, train_accuracy=tr_acc, val_loss=va_loss,
                                    val_accuracy=va_acc, lr_scale=lr_now, epoch_time_sec=time.time() - t0))
        if epoch >= MIN_EPOCHS and meta["patience"] >= PATIENCE:
            meta["stopped"] = True
        meta["epoch"] = epoch

        atomic_save({"model_state": model.state_dict(), "optimizer_state": optimizer.state_dict(),
                     "scheduler_state": scheduler.state_dict(), "scaler_state": scaler.state_dict(),
                     "meta": meta}, paths["latest"])
        pd.DataFrame(meta["history"]).to_csv(paths["history"], index=False)

        flag = "★ best" if improved else "patience " + str(meta["patience"]) + "/" + str(PATIENCE)
        print(f"ep {epoch:02d}/{EPOCHS} | train {tr_loss:.4f}/{tr_acc:.4f} | val {va_loss:.4f}/{va_acc:.4f} "
              f"| lr× {lr_now:.3f} | {(time.time()-t0)/60:.1f} min | {gpu_memory()} | {flag}")
        if meta["stopped"]:
            print(f"■ Early stopping (best epoch {meta['best_epoch']}).")

    # ------------------------------------------------------------- final evaluation (best-on-validation weights)
    best = torch.load(paths["best"], map_location=DEVICE, weights_only=False)
    model.load_state_dict(best["model_state"]); del best
    y_val,  p_val  = predict_probs(model, val_loader,  desc="VAL")
    y_test, p_test = predict_probs(model, test_loader, desc="TEST")
    _,      p_tta  = predict_probs(model, test_loader, tta=True, desc="TEST+TTA")
    m, m_tta, m_val = compute_metrics(y_test, p_test), compute_metrics(y_test, p_tta), compute_metrics(y_val, p_val)

    hist = pd.DataFrame(meta["history"])
    result = dict(tag=tag, model=model_name, seed=seed, train_fraction=frac, n_train=n_train, **m,
                  tta_accuracy=m_tta["accuracy"], tta_macro_f1=m_tta["macro_f1"], tta_ece=m_tta["ece"],
                  val_accuracy=m_val["accuracy"], val_nll=m_val["nll"],
                  best_epoch=meta["best_epoch"], epochs_completed=len(hist),
                  train_time_min=float(hist["epoch_time_sec"].sum() / 60), params_M=n_params / 1e6)
    np.savez_compressed(paths["predictions"], y_true=y_test, probs=p_test, probs_tta=p_tta,
                        val_true=y_val, val_probs=p_val)
    paths["metrics"].write_text(json.dumps(result, indent=2), encoding="utf-8")   # written LAST => marks the run as DONE
    print(f"\n✓ TEST acc {m['accuracy']:.4f} | macro-F1 {m['macro_f1']:.4f} | ECE {m['ece']:.4f} "
          f"| TTA acc {m_tta['accuracy']:.4f}")

    # ------------------------------------------------------------- housekeeping
    if DELETE_LATEST_WHEN_DONE and paths["latest"].exists(): paths["latest"].unlink()
    keep_best = KEEP_BEST_FOR_ANALYSIS and frac >= 1.0 and seed == SEEDS[0]
    if not keep_best and paths["best"].exists(): paths["best"].unlink()
    del model, optimizer, scheduler, scaler, train_loader, val_loader, test_loader
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return result

print("✓ Experiment function ready.")

In [ ]:
# Cell 12 — Experiment manager: plan, status table, run everything (auto-skip / auto-resume)
def build_plan():
    plan = [(m, s, 1.0) for s in SEEDS for m in MAIN_MODELS]
    if RUN_ABLATIONS: plan += [(m, s, 1.0) for s in SEEDS for m in ABLATION_MODELS]
    if RUN_LOW_DATA:  plan += [(m, s, f) for f in LOW_DATA_FRACTIONS for s in LOW_DATA_SEEDS for m in LOW_DATA_MODELS]
    seen, out = set(), []
    for p in plan:
        if p not in seen: seen.add(p); out.append(p)
    return out

def run_status(tag):
    p = get_paths(tag)
    if p["metrics"].exists(): return "DONE"
    if p["history"].exists():  return f"IN PROGRESS (epoch {len(pd.read_csv(p['history']))})"
    return "PENDING"

PLAN = build_plan()
status_df = pd.DataFrame([dict(run=run_tag(m, s, f), model=m, seed=s, fraction=f, status=run_status(run_tag(m, s, f)))
                          for m, s, f in PLAN])
print(f"Planned runs: {len(PLAN)}  |  done: {(status_df.status=='DONE').sum()}  |  "
      f"pending/in-progress: {(status_df.status!='DONE').sum()}")
display(status_df.groupby("status").size().rename("runs").to_frame())

for k, (m, s, f) in enumerate(PLAN, 1):
    tag = run_tag(m, s, f)
    if run_status(tag) == "DONE":
        print(f"[{k}/{len(PLAN)}] {tag}: done — skipped"); continue
    print(f"\n[{k}/{len(PLAN)}] running {tag}")
    try:
        run_single_experiment(m, s, f)
    except torch.cuda.OutOfMemoryError:
        print("CUDA OOM — reduce BATCH_SIZE (keep BATCH_SIZE × GRAD_ACCUM_STEPS = 16 by raising GRAD_ACCUM_STEPS), "
              "then change RUN_ID or restore the fingerprint-relevant settings.")
        raise

def collect_results():
    rows = [json.loads(p.read_text(encoding="utf-8")) for p in sorted(MODEL_ROOT.glob("*_metrics.json"))]
    return pd.DataFrame(rows)

results_df = collect_results()
results_df.to_csv(RUN_ROOT / "all_results.csv", index=False)
print(f"\n✓ {len(results_df)} finished runs collected → all_results.csv")
display(results_df[results_df.train_fraction == 1.0][["tag", "accuracy", "macro_f1", "ece", "tta_accuracy",
                                                       "best_epoch", "epochs_completed", "train_time_min"]].round(4))

## 4 · Main results — accuracy, agreement, calibration, statistical evidence

In [ ]:
# Cell 13 — Result assembly (incl. post-hoc late-fusion baselines) and the main table
results_df = collect_results()
assert len(results_df), "No finished runs yet — run Cell 12 first."
FULL = results_df[results_df.train_fraction == 1.0].copy()

def load_npz(tag): return np.load(get_paths(tag)["predictions"])

def seeds_available(*models):
    return [s for s in SEEDS if all(get_paths(run_tag(m, s))["metrics"].exists() for m in models)]

def late_probs(seed, mode):
    """Post-hoc fusion of the independently trained CNN and ViT: 'avg' (0.5/0.5) or 'val' (weight tuned on validation)."""
    a, b = load_npz(run_tag("cnn", seed)), load_npz(run_tag("vit", seed))
    assert (a["y_true"] == b["y_true"]).all()
    if mode == "avg":
        return a["y_true"], 0.5 * a["probs"] + 0.5 * b["probs"], 0.5
    yv, best = a["val_true"], (-1.0, 1e9, 0.5)
    for w in np.linspace(0, 1, 21):
        pv = w * a["val_probs"] + (1 - w) * b["val_probs"]
        acc = (pv.argmax(1) == yv).mean()
        nll = -np.log(np.clip(pv[np.arange(len(yv)), yv], 1e-12, 1)).mean()
        if acc > best[0] + 1e-12 or (abs(acc - best[0]) <= 1e-12 and nll < best[1]): best = (acc, nll, w)
    w = best[2]
    return a["y_true"], w * a["probs"] + (1 - w) * b["probs"], w

def get_probs(model, seed):
    if model == "late_avg": return late_probs(seed, "avg")[:2]
    if model == "late_val": return late_probs(seed, "val")[:2]
    d = load_npz(run_tag(model, seed)); return d["y_true"], d["probs"]

late_rows = []
for s in seeds_available("cnn", "vit"):
    for key, mode in [("late_avg", "avg"), ("late_val", "val")]:
        y, p, w = late_probs(s, mode)
        late_rows.append(dict(model=key, seed=s, train_fraction=1.0, params_M=np.nan, cnn_weight=w, **compute_metrics(y, p)))
ALL_FULL = pd.concat([FULL, pd.DataFrame(late_rows)], ignore_index=True)

METRIC_SPECS = [("accuracy", "Accuracy (%)", 100, 2), ("macro_f1", "Macro-F1 (%)", 100, 2),
                ("mcc", "MCC", 1, 4), ("cohen_kappa", "Kappa", 1, 4),
                ("roc_auc_ovr_macro", "AUC (%)", 100, 2), ("ece", "ECE (%)", 100, 2), ("nll", "NLL", 1, 3)]
LOWER_IS_BETTER = {"ece", "nll", "brier"}

def mean_sd(x, scale, dec):
    x = np.asarray(x, float) * scale
    sd = x.std(ddof=1) if len(x) > 1 else 0.0
    return f"{x.mean():.{dec}f} ± {sd:.{dec}f}"

def make_table(df, order, specs=METRIC_SPECS, extra_cols=True):
    rows, num = [], []
    for m in order:
        g = df[df.model == m]
        if not len(g): continue
        r = {"Model": LABEL[m], "Seeds": len(g)}; n = {"model": m}
        for key, name, sc, dec in specs:
            r[name] = mean_sd(g[key], sc, dec); n[key] = g[key].mean()
        if extra_cols and m in ("cnn", "vit") or (extra_cols and m in FUSION_CONFIGS):
            r["Params (M)"] = f"{g['params_M'].mean():.1f}"
        elif extra_cols:
            r["Params (M)"] = "cnn+vit"
        rows.append(r); num.append(n)
    return pd.DataFrame(rows), pd.DataFrame(num)

def _esc(s): return str(s).replace("%", r"\%").replace("±", r"$\pm$").replace("Δ", r"$\Delta$").replace("_", r"\_").replace("&", r"\&")

def write_latex(df, num_df, specs, path, caption, label):
    cols = list(df.columns)
    best = {}
    for key, name, sc, dec in specs:
        v = num_df[key].values
        best[name] = int(np.nanargmin(v) if key in LOWER_IS_BETTER else np.nanargmax(v))
    L = [r"\begin{table}[t]", r"\centering", r"\small", rf"\caption{{{caption}}}", rf"\label{{{label}}}",
         r"\begin{tabular}{l" + "c" * (len(cols) - 1) + "}", r"\toprule", " & ".join(_esc(c) for c in cols) + r" \\", r"\midrule"]
    for i, row in df.iterrows():
        cells = []
        for c in cols:
            t = _esc(row[c]); cells.append(rf"\textbf{{{t}}}" if best.get(c) == i else t)
        L.append(" & ".join(cells) + r" \\")
    L += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    Path(path).write_text("\n".join(L), encoding="utf-8")

MAIN_ORDER = ["cnn", "vit", "late_avg", "late_val", "concat", "gate_scalar", "cgf"]
main_tbl, main_num = make_table(ALL_FULL, MAIN_ORDER)
main_tbl.to_csv(TABLE_ROOT / "table_main_results.csv", index=False)
write_latex(main_tbl, main_num, METRIC_SPECS, TABLE_ROOT / "table_main_results.tex",
            "Test-set performance on AID (mean $\\pm$ SD over seeds). Best value per column in bold.", "tab:main")
print("TEST-SET RESULTS (mean ± SD across seeds; late-fusion rows are post-hoc combinations of the trained CNN and ViT)")
display(main_tbl)

tta_tbl = ALL_FULL[ALL_FULL.model.isin(["cnn", "vit", "concat", "gate_scalar", "cgf"])].groupby("model")[
    ["accuracy", "tta_accuracy", "macro_f1", "tta_macro_f1", "ece", "tta_ece"]].mean().reindex(
    [m for m in ["cnn", "vit", "concat", "gate_scalar", "cgf"] if m in ALL_FULL.model.values])
tta_tbl.index = [LABEL[m] for m in tta_tbl.index]
tta_tbl.round(4).to_csv(TABLE_ROOT / "table_tta.csv")
print("\nEffect of dihedral (8-view) test-time augmentation — extra rows, not used in the main comparison:")
display((tta_tbl * 100).round(2))

In [ ]:
# Cell 14 — Statistical evidence: exact McNemar per seed (Holm-corrected) + paired bootstrap CI over images
from scipy.stats import binomtest

REF = "cgf"
cands = [m for m in ["cnn", "vit", "late_avg", "late_val", "concat", "gate_scalar"]
         if len(seeds_available(*(["cnn", "vit"] if m.startswith("late") else [m]))) == len(seeds_available(REF)) > 0]
seeds_ok = seeds_available(REF, *[c for c in cands if not c.startswith("late")], "cnn", "vit")

def holm(p):
    p = np.asarray(p); order = np.argsort(p); adj = np.empty_like(p, dtype=float); run = 0.0
    for rank, idx in enumerate(order):
        run = max(run, (len(p) - rank) * p[idx]); adj[idx] = min(1.0, run)
    return adj

correct = {}
for m in [REF] + cands:
    arr = []
    for s in seeds_ok:
        y, p = get_probs(m, s); arr.append(p.argmax(1) == y)
    correct[m] = np.stack(arr)                                     # (seeds, N)

mc_rows = []
for si, s in enumerate(seeds_ok):
    pvals, tmp = [], []
    for m in cands:
        a, b = correct[REF][si], correct[m][si]
        n10, n01 = int((a & ~b).sum()), int((~a & b).sum())        # REF right & other wrong / REF wrong & other right
        p = binomtest(min(n10, n01), n10 + n01, 0.5).pvalue if n10 + n01 > 0 else 1.0
        pvals.append(p); tmp.append(dict(seed=s, comparison=f"{LABEL[REF]} vs {LABEL[m]}", other=m,
                                         acc_ref=a.mean() * 100, acc_other=b.mean() * 100,
                                         only_ref_correct=n10, only_other_correct=n01, p_exact=p))
    for r, adj in zip(tmp, holm(pvals)): r["p_holm"] = adj
    mc_rows += tmp
mcnemar_df = pd.DataFrame(mc_rows)
mcnemar_df.round(5).to_csv(TABLE_ROOT / "table_mcnemar_per_seed.csv", index=False)

rng = np.random.RandomState(0); B = 5000; N = correct[REF].shape[1]
ci_rows = []
for m in cands:
    d = correct[REF].mean(0) - correct[m].mean(0)                  # per-image paired difference, averaged over seeds
    boots = np.array([d[rng.randint(0, N, N)].mean() for _ in range(B)]) * 100
    sub = mcnemar_df[mcnemar_df.other == m]
    ci_rows.append({"Comparison": f"{LABEL[REF]} − {LABEL[m]}", "Δ accuracy (pp)": f"{d.mean()*100:+.2f}",
                    "95% bootstrap CI": f"[{np.percentile(boots, 2.5):+.2f}, {np.percentile(boots, 97.5):+.2f}]",
                    "Seeds with Holm-p<0.05": f"{int((sub.p_holm < 0.05).sum())}/{len(sub)}",
                    "median p (Holm)": f"{sub.p_holm.median():.3g}"})
sig_tbl = pd.DataFrame(ci_rows)
sig_tbl.to_csv(TABLE_ROOT / "table_significance.csv", index=False)
print(f"Paired comparisons of {LABEL[REF]} against each alternative on the same {N} test images "
      f"({len(seeds_ok)} seeds; bootstrap resamples: {B}).")
display(sig_tbl)
print("Interpretation guide: a CI excluding 0 supports a real difference on this test set; McNemar p-values are per seed "
      "(seeds are not independent samples of the test set, so they are reported, not pooled).")

In [ ]:
# Cell 15 — Branch complementarity: does fusion recover what each single backbone misses?
comp_rows = []
for s in seeds_available("cnn", "vit", REF):
    y, pc = get_probs("cnn", s); _, pv = get_probs("vit", s); _, pf = get_probs(REF, s)
    c, v, f = pc.argmax(1) == y, pv.argmax(1) == y, pf.argmax(1) == y
    cats = {"Both backbones correct": c & v, "Only CNN correct": c & ~v,
            "Only ViT correct": ~c & v, "Both backbones wrong": ~c & ~v}
    row = {"seed": s, "oracle_accuracy(CNN or ViT)": (c | v).mean()}
    for name, mask in cats.items():
        row[f"{name} — share"] = mask.mean()
        row[f"{name} — CGF acc"] = f[mask].mean() if mask.any() else np.nan
    comp_rows.append(row)
comp_df = pd.DataFrame(comp_rows)
comp_mean = comp_df.drop(columns="seed").mean()
comp_out = pd.DataFrame({
    "Subset": ["Both backbones correct", "Only CNN correct", "Only ViT correct", "Both backbones wrong"],
    "Share of test images (%)": [comp_mean[f"{k} — share"] * 100 for k in
                                 ["Both backbones correct", "Only CNN correct", "Only ViT correct", "Both backbones wrong"]],
    "CGF-Net accuracy on subset (%)": [comp_mean[f"{k} — CGF acc"] * 100 for k in
                                       ["Both backbones correct", "Only CNN correct", "Only ViT correct", "Both backbones wrong"]]}).round(2)
comp_out.to_csv(TABLE_ROOT / "table_complementarity.csv", index=False)
display(comp_out)
print(f"Oracle accuracy (an image counts as solved if either backbone solves it): {comp_mean['oracle_accuracy(CNN or ViT)']*100:.2f}% "
      f"— an upper bound for any decision-level combination of these two backbones.")

In [ ]:
# Cell 16 — Calibration: reliability diagram (seeds pooled)
def reliability(probs, y, n_bins=10):
    conf, pred = probs.max(1), probs.argmax(1); acc = (pred == y).astype(float)
    edges = np.linspace(0, 1, n_bins + 1); xs, ys, ns = [], [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() >= 5: xs.append(conf[m].mean()); ys.append(acc[m].mean()); ns.append(int(m.sum()))
    return np.array(xs), np.array(ys), np.array(ns)

fig, axes = plt.subplots(1, 2, figsize=(COL2, 2.5), layout="constrained")
axes[0].plot([0, 1], [0, 1], "k--", lw=0.6, label="Perfect calibration")
for m in ["cnn", "vit", "concat", "cgf"]:
    ss = seeds_available(m)
    if not ss: continue
    y = np.concatenate([get_probs(m, s)[0] for s in ss]); p = np.concatenate([get_probs(m, s)[1] for s in ss])
    x_, y_, _ = reliability(p, y)
    axes[0].plot(x_, y_, marker=MARKER[m], color=COLOR[m], label=LABEL[m], ms=3.5 if m != "cgf" else 5)
    axes[1].hist(p.max(1), bins=np.linspace(0, 1, 26), histtype="step", color=COLOR[m], lw=1.0, label=LABEL[m], density=True)
axes[0].set_xlabel("Confidence"); axes[0].set_ylabel("Accuracy"); axes[0].set_title("(a) Reliability diagram")
axes[0].legend(loc="upper left"); axes[0].set_xlim(0.0, 1.0); axes[0].set_ylim(0.0, 1.02)
axes[1].set_xlabel("Max softmax probability"); axes[1].set_ylabel("Density (log)"); axes[1].set_yscale("log")
axes[1].set_title("(b) Confidence distribution")
save_fig(fig, "fig03_calibration")

## 5 · Learning behaviour, error structure and interpretability of the fusion

In [ ]:
# Cell 17 — Learning curves (mean ± SD over seeds)
def load_histories(model):
    out = []
    for s in SEEDS:
        f = get_paths(run_tag(model, s))["history"]
        if f.exists(): out.append(pd.read_csv(f))
    return out

def stack_metric(hists, col):
    L = max(len(h) for h in hists); A = np.full((len(hists), L), np.nan)
    for i, h in enumerate(hists): A[i, :len(h)] = h[col].values
    return A

fig, axes = plt.subplots(1, 3, figsize=(COL2, 2.35), layout="constrained")
spec = [("val_accuracy", "Validation accuracy", "(a)"), ("val_loss", "Validation loss (CE)", "(b)"), ("train_accuracy", "Training accuracy", "(c)")]
acc_floor = []
for ax, (col, title, tag_) in zip(axes, spec):
    for m in MAIN_MODELS:
        hs = load_histories(m)
        if not hs: continue
        A = stack_metric(hs, col); mu = np.nanmean(A, 0); sd = np.nanstd(A, 0); ep = np.arange(1, len(mu) + 1)
        ax.plot(ep, mu, color=COLOR[m], label=LABEL[m], lw=1.4 if m == "cgf" else 0.9)
        ax.fill_between(ep, mu - sd, mu + sd, color=COLOR[m], alpha=0.15, lw=0)
        if col == "val_accuracy" and len(mu) > 3: acc_floor.append(mu[2])
    ax.set_xlabel("Epoch"); ax.set_title(f"{tag_} {title}")
if acc_floor:
    axes[0].set_ylim(max(0, min(acc_floor) - 0.03), 1.0)
axes[1].set_yscale("log")
axes[0].legend(loc="lower right", fontsize=5.6)
save_fig(fig, "fig04_learning_curves")

In [ ]:
# Cell 18 — Confusion structure (CGF-Net, seeds pooled) and per-class F1 across models
cls_labels = list(range(NUM_CLASSES))
ss = seeds_available(REF)
yc = np.concatenate([get_probs(REF, s)[0] for s in ss]); pc = np.concatenate([get_probs(REF, s)[1] for s in ss]).argmax(1)
cm = confusion_matrix(yc, pc, labels=cls_labels); cmn = cm / cm.sum(1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(COL2, 6.3), layout="constrained")
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=100, interpolation="nearest")
ax.set_xticks(cls_labels); ax.set_yticks(cls_labels)
ax.set_xticklabels(classes, rotation=90, fontsize=5.2); ax.set_yticklabels(classes, fontsize=5.2)
ax.set_xlabel("Predicted class"); ax.set_ylabel("True class"); ax.grid(False)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if i != j and cmn[i, j] >= 3.0:
            ax.text(j, i, f"{cmn[i, j]:.0f}", ha="center", va="center", fontsize=4.3, color="white" if cmn[i, j] > 55 else "black")
cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.01); cb.set_label("Row-normalised (%)", fontsize=6); cb.ax.tick_params(labelsize=5.5)
ax.set_title(f"{LABEL[REF]} — confusion matrix pooled over {len(ss)} seeds (off-diagonal cells ≥ 3 % annotated)", fontsize=7)
save_fig(fig, "fig05_confusion_matrix_cgf")

off = [(cm[i, j], i, j) for i in range(NUM_CLASSES) for j in range(NUM_CLASSES) if i != j and cm[i, j] > 0]
top = sorted(off, reverse=True)[:10]
conf_tbl = pd.DataFrame([{"True class": classes[i], "Predicted as": classes[j], "Count (all seeds)": int(n),
                          "Rate (% of true class)": round(100 * n / cm[i].sum(), 2)} for n, i, j in top])
conf_tbl.to_csv(TABLE_ROOT / "table_top_confusions_cgf.csv", index=False)
print("Most frequent confusions:"); display(conf_tbl)

# ---- per-class F1 (mean over seeds) ---------------------------------------
order_pc = [m for m in ["cnn", "vit", "concat", "gate_scalar", "cgf"] if seeds_available(m)]
f1_tab = {}
for m in order_pc:
    vals = []
    for s in seeds_available(m):
        y, p = get_probs(m, s); vals.append(f1_score(y, p.argmax(1), labels=cls_labels, average=None, zero_division=0))
    f1_tab[m] = np.mean(vals, 0)
f1_df = pd.DataFrame(f1_tab, index=classes)
f1_df.round(4).to_csv(TABLE_ROOT / "table_per_class_f1.csv")
f1_sorted = f1_df.sort_values(REF, ascending=True)

fig, ax = plt.subplots(figsize=(COL1, 5.6), layout="constrained")
f1_vmin = max(0.5, float(f1_sorted.values.min()) - 0.02)
im = ax.imshow(f1_sorted.values, cmap="YlGnBu", vmin=f1_vmin, vmax=1.0, aspect="auto")
ax.set_xticks(range(len(order_pc))); ax.set_xticklabels([LABEL[m].replace(" (proposed)", "\n(proposed)").replace("fusion", "\nfusion") for m in order_pc], fontsize=5.3, rotation=0)
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(f1_sorted.index, fontsize=5.4); ax.grid(False)
ax.xaxis.tick_top()
for i in range(NUM_CLASSES):
    for j in range(len(order_pc)):
        v = f1_sorted.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=4.6, color="white" if v > f1_vmin + 0.62 * (1 - f1_vmin) else "black")
fig.colorbar(im, ax=ax, fraction=0.05, pad=0.02).set_label("F1", fontsize=6)
save_fig(fig, "fig06_per_class_f1")

In [ ]:
# Cell 19 — Gate behaviour of CGF-Net and embedding geometry (uses best checkpoint of SEEDS[0])
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

def load_trained(name, seed=None):
    seed = SEEDS[0] if seed is None else seed
    ck_path = get_paths(run_tag(name, seed))["best"]
    if not ck_path.exists(): return None
    m = build_model(name, pretrained=False)
    ck = torch.load(ck_path, map_location="cpu", weights_only=False)
    m.load_state_dict(ck["model_state"]); return m.to(DEVICE).eval()

_, _, analysis_loader, _ = make_loaders(SEEDS[0])

# ---------------- (A) gate analysis --------------------------------------------------------------------------
m = load_trained("cgf")
if m is None:
    print("best.pth of CGF-Net (SEEDS[0]) not found — gate analysis skipped.")
else:
    Y, G, CC, VC, FC = [], [], [], [], []
    with torch.no_grad():
        for images, labels in tqdm(analysis_loader, desc="gate analysis", leave=False):
            with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
                out = m(images.to(DEVICE), return_all=True)
            Y.append(labels.numpy()); G.append(out["gate"].float().cpu().numpy())
            CC.append(out["cnn_logits"].argmax(1).cpu().numpy()); VC.append(out["vit_logits"].argmax(1).cpu().numpy())
            FC.append(out["logits"].argmax(1).cpu().numpy())
    Y, G, CC, VC, FC = map(np.concatenate, (Y, G, CC, VC, FC))
    g_mean = G.mean(1)                                         # average CNN weight per image (over the 256 channels)
    c_ok, v_ok = CC == Y, VC == Y
    groups = {"Only CNN branch correct": c_ok & ~v_ok, "Only ViT branch correct": ~c_ok & v_ok,
              "Both branches correct": c_ok & v_ok, "Both branches wrong": ~c_ok & ~v_ok}
    gate_tbl = pd.DataFrame([{"Subset": k, "Images": int(v.sum()),
                              "Mean CNN weight": round(float(g_mean[v].mean()), 4) if v.any() else np.nan} for k, v in groups.items()])
    gate_tbl.to_csv(TABLE_ROOT / "table_gate_by_subset.csv", index=False)
    print("Does the gate lean on the branch that is actually right?"); display(gate_tbl)

    per_class = pd.Series(g_mean).groupby(Y).agg(["mean", "std"]); per_class.index = classes
    per_class = per_class.sort_values("mean")
    chan_mean = G.mean(0)
    fig, axes = plt.subplots(1, 2, figsize=(COL2, 3.5), gridspec_kw=dict(width_ratios=[1.25, 1]), layout="constrained")
    axes[0].barh(range(NUM_CLASSES), per_class["mean"], xerr=per_class["std"], color=COLOR["cgf"], error_kw=dict(lw=0.5, capsize=1))
    axes[0].axvline(0.5, color="k", lw=0.6, ls="--")
    axes[0].set_yticks(range(NUM_CLASSES)); axes[0].set_yticklabels(per_class.index, fontsize=5.3)
    axes[0].set_xlabel("Mean CNN weight (1 − ViT weight)"); axes[0].set_title("(a) Per-class reliance on the CNN branch")
    axes[0].grid(axis="y", visible=False)
    axes[1].hist(chan_mean, bins=30, color=COLOR["cgf"], alpha=0.85)
    axes[1].axvline(0.5, color="k", lw=0.6, ls="--")
    axes[1].set_xlabel("Per-channel mean gate value"); axes[1].set_ylabel("Feature channels")
    axes[1].set_title(f"(b) Channel specialisation (d={G.shape[1]})")
    save_fig(fig, "fig07_gate_analysis")
    print(f"Channels leaning CNN (>0.5): {(chan_mean > 0.5).sum()} / {len(chan_mean)}; "
          f"std of per-channel means: {chan_mean.std():.3f} (a scalar gate cannot express this spread)")
    del m; gc.collect()

# ---------------- (B) embedding geometry: t-SNE + silhouette ---------------------------------------------------
emb = {}
for name in ["cnn", "vit", "cgf"]:
    model = load_trained(name)
    if model is None: continue
    feats, ys = [], []
    with torch.no_grad():
        for images, labels in tqdm(analysis_loader, desc=f"embeddings {name}", leave=False):
            with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
                feats.append(model.embed(images.to(DEVICE)).float().cpu().numpy())
            ys.append(labels.numpy())
    emb[name] = (np.concatenate(feats), np.concatenate(ys)); del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

if emb:
    cmap = matplotlib.colormaps["nipy_spectral"]; cols = [cmap(i / (NUM_CLASSES - 1)) for i in range(NUM_CLASSES)]
    fig, axes = plt.subplots(1, len(emb), figsize=(COL2, 2.5), layout="constrained")
    axes = np.atleast_1d(axes); sil_rows = []
    for ax, (name, (X, y)) in zip(axes, emb.items()):
        Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
        perp = min(30, max(5, (len(Xn) - 1) // 3))
        Z = TSNE(n_components=2, perplexity=perp, init="pca", learning_rate="auto", random_state=0).fit_transform(Xn)
        sil = silhouette_score(Xn, y)
        ax.scatter(Z[:, 0], Z[:, 1], c=[cols[i] for i in y], s=1.6, linewidths=0, rasterized=True)
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        ax.set_title(f"{LABEL[name]}\nsilhouette = {sil:.3f}", fontsize=6.8)
        sil_rows.append(dict(model=LABEL[name], silhouette_cosine=round(float(sil), 4)))
    save_fig(fig, "fig08_tsne_embeddings")
    pd.DataFrame(sil_rows).to_csv(TABLE_ROOT / "table_silhouette.csv", index=False); display(pd.DataFrame(sil_rows))
del analysis_loader

## 6 · Stress tests: robustness, label efficiency, compute cost, ablations

In [ ]:
# Cell 20 — Robustness to image degradations (identical perturbations for every model; SEEDS[0] checkpoints)
from torchvision.transforms import functional as TF

mean_t = torch.tensor(IMAGENET_MEAN, device=DEVICE).view(1, 3, 1, 1)
std_t  = torch.tensor(IMAGENET_STD,  device=DEVICE).view(1, 3, 1, 1)
denorm = lambda x: x * std_t + mean_t
renorm = lambda x: (x - mean_t) / std_t

def p_noise(s):  return lambda x, rng, gen: (x + s * torch.randn(x.shape, generator=gen).to(x.device)).clamp(0, 1)
def p_blur(k, s): return lambda x, rng, gen: TF.gaussian_blur(x, kernel_size=[k, k], sigma=[s, s])
def p_lowres(f):
    return lambda x, rng, gen: F.interpolate(F.interpolate(x, scale_factor=1 / f, mode="bilinear", antialias=True, align_corners=False),
                                             size=x.shape[-2:], mode="bilinear", align_corners=False)
def p_bright(f): return lambda x, rng, gen: (x * f).clamp(0, 1)
def p_contrast(f):
    return lambda x, rng, gen: ((x - x.mean((1, 2, 3), keepdim=True)) * f + x.mean((1, 2, 3), keepdim=True)).clamp(0, 1)
def p_occlude(frac):
    def fn(x, rng, gen):
        x = x.clone(); H, W = x.shape[-2:]; side = int(round(math.sqrt(frac) * H))
        for i in range(x.shape[0]):
            r, c = rng.randint(0, H - side + 1), rng.randint(0, W - side + 1); x[i, :, r:r + side, c:c + side] = 0.5
        return x
    return fn

PERTURBATIONS = {
    "Clean": lambda x, rng, gen: x,
    "Noise σ=0.05": p_noise(0.05), "Noise σ=0.10": p_noise(0.10),
    "Blur σ=2": p_blur(9, 2.0), "Blur σ=4": p_blur(17, 4.0),
    "Low-res ÷4": p_lowres(4), "Low-res ÷8": p_lowres(8),
    "Dark ×0.5": p_bright(0.5), "Bright ×1.6": p_bright(1.6), "Contrast ×0.4": p_contrast(0.4),
    "Occlusion 25%": p_occlude(0.25),
}

rob_models = [m for m in MAIN_MODELS if get_paths(run_tag(m, SEEDS[0]))["best"].exists()]
if not RUN_ROBUSTNESS or not rob_models:
    print("Robustness skipped (disabled or no best.pth available).")
else:
    _, _, rob_loader, _ = make_loaders(SEEDS[0])
    rob = {}
    for name in rob_models:
        model = load_trained(name)
        for pname, pfn in PERTURBATIONS.items():
            rng_, gen_ = np.random.RandomState(0), torch.Generator().manual_seed(0)   # same perturbation for all models
            y, p = predict_probs(model, rob_loader, desc=f"{name} | {pname}",
                                 transform_fn=lambda x: renorm(pfn(denorm(x), rng_, gen_)))
            rob[(name, pname)] = float((p.argmax(1) == y).mean() * 100)
        del model; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    del rob_loader

    rob_df = pd.DataFrame({LABEL[m]: [rob[(m, pn)] for pn in PERTURBATIONS] for m in rob_models}, index=list(PERTURBATIONS)).T
    rob_df.round(2).to_csv(TABLE_ROOT / "table_robustness_accuracy.csv")
    print("Test accuracy (%) under perturbation (single seed = SEEDS[0]; differences of a few tenths of a point are within seed noise):")
    display(rob_df.round(2))

    conds = [c for c in PERTURBATIONS if c != "Clean"]
    fig, ax = plt.subplots(figsize=(COL2, 2.7), layout="constrained")
    w = 0.8 / len(rob_models)
    for k, m in enumerate(rob_models):
        drops = [rob[(m, "Clean")] - rob[(m, c)] for c in conds]
        ax.bar(np.arange(len(conds)) + (k - (len(rob_models) - 1) / 2) * w, drops, w, color=COLOR[m], label=LABEL[m], lw=0)
    ax.set_xticks(range(len(conds))); ax.set_xticklabels(conds, rotation=30, ha="right", fontsize=6)
    ax.set_ylabel("Accuracy drop vs. clean (pp) — lower is better"); ax.grid(axis="x", visible=False)
    ax.legend(ncol=len(rob_models), loc="upper left", fontsize=5.6, columnspacing=0.8, handlelength=1.0)
    save_fig(fig, "fig09_robustness")

In [ ]:
# Cell 21 — Compute cost: parameters, FLOPs, latency, throughput, and accuracy-vs-cost
def profile_model(name, bs_thr=None):
    bs_thr = bs_thr or EVAL_BATCH_SIZE
    m = build_model(name, pretrained=False).to(DEVICE).eval()
    x1 = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
    gflops = np.nan
    try:
        from torch.utils.flop_counter import FlopCounterMode
        with torch.no_grad(), FlopCounterMode(display=False) as fc: m(x1)
        gflops = fc.get_total_flops() / 1e9
    except Exception as e:
        print(f"FLOP counter unavailable for {name}: {type(e).__name__}")
    def timeit(x, n_warm=8, n_run=25):
        with torch.no_grad(), torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
            for _ in range(n_warm): m(x)
            if torch.cuda.is_available(): torch.cuda.synchronize()
            t0 = time.perf_counter()
            for _ in range(n_run): m(x)
            if torch.cuda.is_available(): torch.cuda.synchronize()
        return (time.perf_counter() - t0) / n_run
    lat = timeit(x1) * 1000
    xb = torch.randn(bs_thr, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
    thr = bs_thr / timeit(xb, 3, 8)
    out = dict(model=name, params_M=count_params(m) / 1e6, gflops=gflops, latency_ms_bs1=lat, throughput_img_s=thr)
    del m; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return out

prof = pd.DataFrame([profile_model(n) for n in MAIN_MODELS + [a for a in ABLATION_MODELS if RUN_ABLATIONS]]).set_index("model")
acc_mean = ALL_FULL.groupby("model")["accuracy"].agg(["mean", "std"]) * 100
prof["accuracy_mean"], prof["accuracy_sd"] = acc_mean["mean"], acc_mean["std"]
prof["train_min_per_run"] = FULL.groupby("model")["train_time_min"].mean()
# late fusion pays for both networks
extra = {}
if {"cnn", "vit"} <= set(prof.index):
    for key in ["late_avg", "late_val"]:
        if key in acc_mean.index:
            c_, v_ = prof.loc["cnn"], prof.loc["vit"]
            extra[key] = dict(params_M=c_["params_M"] + v_["params_M"], gflops=c_["gflops"] + v_["gflops"],
                              latency_ms_bs1=c_["latency_ms_bs1"] + v_["latency_ms_bs1"],
                              throughput_img_s=1.0 / (1.0 / c_["throughput_img_s"] + 1.0 / v_["throughput_img_s"]),
                              accuracy_mean=acc_mean.loc[key, "mean"], accuracy_sd=acc_mean.loc[key, "std"])
if extra:
    prof = pd.concat([prof, pd.DataFrame(extra).T])
prof.round(3).to_csv(TABLE_ROOT / "table_efficiency.csv")
disp = prof.copy(); disp.index = [LABEL[i] for i in disp.index]
display(disp.round(2))
print("FLOPs = 2×MACs of one forward pass at 224×224 (torch.utils.flop_counter). Latency: batch 1; throughput: "
      f"batch {EVAL_BATCH_SIZE}; AMP {'on' if USE_AMP else 'off'}; device {DEVICE}.")

fig, ax = plt.subplots(figsize=(COL1, 2.6), layout="constrained")
for m, r in prof.dropna(subset=["gflops", "accuracy_mean"]).iterrows():
    if m not in COLOR or (m in ABLATION_MODELS): continue
    ax.errorbar(r["gflops"], r["accuracy_mean"], yerr=r["accuracy_sd"] if not np.isnan(r["accuracy_sd"]) else 0,
                fmt=MARKER[m], color=COLOR[m], ms=6 if m == "cgf" else 4.5, capsize=1.5, elinewidth=0.6, label=LABEL[m])
ax.set_xlabel("GFLOPs per image (forward)"); ax.set_ylabel("Test accuracy (%)")
ax.legend(fontsize=5.4, loc="lower right", handletextpad=0.2)
save_fig(fig, "fig10_accuracy_vs_cost")

In [ ]:
# Cell 22 — Label efficiency (nested training subsets) and component ablation
# ---------------------------------------------------------------- (A) low-data study
if RUN_LOW_DATA:
    rows = []
    for m in LOW_DATA_MODELS:
        for f in sorted(LOW_DATA_FRACTIONS) + [1.0]:
            g = results_df[(results_df.model == m) & (np.isclose(results_df.train_fraction, f)) & (results_df.seed.isin(LOW_DATA_SEEDS))]
            if len(g):
                rows.append(dict(model=m, fraction=f, n_train=int(g.n_train.iloc[0]), pct_of_dataset=f * TRAIN_RATIO * 100,
                                 accuracy=g.accuracy.mean() * 100, macro_f1=g.macro_f1.mean() * 100, ece=g.ece.mean() * 100, n=len(g)))
    ld = pd.DataFrame(rows)
    if len(ld):
        ld.round(3).to_csv(TABLE_ROOT / "table_label_efficiency.csv", index=False)
        piv = ld.pivot(index="model", columns="n_train", values="accuracy").reindex(LOW_DATA_MODELS)
        piv.index = [LABEL[i] for i in piv.index]
        print(f"Test accuracy (%) vs number of training images (seeds: {LOW_DATA_SEEDS}):"); display(piv.round(2))
        fig, axes = plt.subplots(1, 2, figsize=(COL2 * 0.62, 2.4), layout="constrained")
        for ax, col, title in zip(axes, ["accuracy", "macro_f1"], ["(a) Accuracy (%)", "(b) Macro-F1 (%)"]):
            for m in LOW_DATA_MODELS:
                d = ld[ld.model == m].sort_values("fraction")
                ax.plot(d.pct_of_dataset, d[col], marker=MARKER[m], color=COLOR[m], label=LABEL[m], ms=4.5 if m != "cgf" else 6)
            ax.set_xlabel("Training data (% of AID)"); ax.set_title(title); ax.set_xticks(sorted(ld.pct_of_dataset.unique()))
        axes[0].legend(loc="lower right", fontsize=5.6)
        save_fig(fig, "fig11_label_efficiency")
else:
    print("Low-data study disabled.")

# ---------------------------------------------------------------- (B) ablation
abl_order = ["gate_scalar", "concat", "cgf_noattn", "cgf_nogate", "cgf_noaux", "cgf"]
abl_order = [m for m in abl_order if m in ALL_FULL.model.values]
abl_specs = [("accuracy", "Accuracy (%)", 100, 2), ("macro_f1", "Macro-F1 (%)", 100, 2), ("ece", "ECE (%)", 100, 2), ("nll", "NLL", 1, 3)]
abl_tbl, abl_num = make_table(ALL_FULL, abl_order, abl_specs)
ref_acc = ALL_FULL[ALL_FULL.model == "cgf"].set_index("seed")["accuracy"] * 100
abl_tbl.insert(2, "Cross-attn", ["✓" if FUSION_CONFIGS[m]["xattn"] else "–" for m in abl_order if m in FUSION_CONFIGS])
abl_tbl.insert(3, "Channel gate", ["✓" if FUSION_CONFIGS[m]["kind"] == "channel_gate" else ("scalar" if FUSION_CONFIGS[m]["kind"] == "scalar_gate" else "–") for m in abl_order])
abl_tbl.insert(4, "Aux loss", ["✓" if FUSION_CONFIGS[m]["aux"] else "–" for m in abl_order])
dvals = []
for m in abl_order:
    a = ALL_FULL[ALL_FULL.model == m].set_index("seed")["accuracy"] * 100
    dvals.append(f"{(a - ref_acc).dropna().mean():+.2f}" if m != "cgf" else "—")
abl_tbl["Δ Acc vs CGF (pp)"] = dvals
abl_tbl.to_csv(TABLE_ROOT / "table_ablation.csv", index=False)
write_latex(abl_tbl.replace("✓", r"\checkmark").replace("–", "--"), abl_num, abl_specs, TABLE_ROOT / "table_ablation.tex",
            "Ablation of CGF-Net components (mean $\\pm$ SD over seeds).", "tab:ablation")
display(abl_tbl)

fig, ax = plt.subplots(figsize=(COL1, 2.4), layout="constrained")
for k, m in enumerate(abl_order):
    g = ALL_FULL[ALL_FULL.model == m]["accuracy"].values * 100
    ax.scatter(g, np.full(len(g), k) + np.linspace(-0.12, 0.12, len(g)), s=9, color=COLOR[m], alpha=0.75, zorder=3, lw=0)
    ax.errorbar(g.mean(), k, xerr=g.std(ddof=1) if len(g) > 1 else 0, fmt="D", color=COLOR[m], mec="k", mew=0.4, ms=4, capsize=2, elinewidth=0.7, zorder=4)
ax.set_yticks(range(len(abl_order))); ax.set_yticklabels([LABEL[m] for m in abl_order], fontsize=6)
ax.set_xlabel("Test accuracy (%) — dots: seeds, diamond: mean ± SD"); ax.grid(axis="y", visible=False); ax.invert_yaxis()
save_fig(fig, "fig12_ablation")

## 7 · Export for the manuscript

In [ ]:
# Cell 23 — Export: figure manifest, markdown summary of all tables, file inventory
pd.DataFrame(FIG_LOG).drop_duplicates("figure", keep="last").to_csv(RUN_ROOT / "FIGURE_MANIFEST.csv", index=False)
print("FIGURE MANIFEST (every figure is created at its final print size; place it without any resizing)")
display(pd.DataFrame(FIG_LOG).drop_duplicates("figure", keep="last"))

def md_table(df):
    cols = [str(c) for c in df.columns]
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for _, r in df.iterrows(): lines.append("| " + " | ".join(str(v) for v in r.values) + " |")
    return "\n".join(lines)

parts = [f"# Results summary — {RUN_ID}", f"Config fingerprint: `{FINGERPRINT}` · seeds: {SEEDS} · epochs ≤ {EPOCHS} · "
         f"device: {DEVICE}\n", "## Main results (test set, mean ± SD over seeds)", md_table(main_tbl)]
for title, fname in [("Statistical comparison against CGF-Net", "table_significance.csv"), ("Ablation", "table_ablation.csv"),
                     ("Branch complementarity", "table_complementarity.csv"), ("Gate by subset", "table_gate_by_subset.csv"),
                     ("Label efficiency", "table_label_efficiency.csv"), ("Efficiency", "table_efficiency.csv")]:
    f = TABLE_ROOT / fname
    if f.exists(): parts += [f"\n## {title}", md_table(pd.read_csv(f).round(3))]
(RUN_ROOT / "RESULTS_SUMMARY.md").write_text("\n".join(parts), encoding="utf-8")

print("\nFiles written under", RUN_ROOT)
for folder in [TABLE_ROOT, FIG_ROOT]:
    files = sorted(folder.glob("*"))
    print(f"\n{folder.name}/  ({len(files)} files, {sum(f.stat().st_size for f in files)/1e6:.1f} MB)")
    for f in files: print(f"  {f.name:48s} {f.stat().st_size/1024:8.0f} KB")
print("\n✓ Study complete. See RESULTS_SUMMARY.md, TABLES/ and FIGURES/.")

## Manuscript map (fill with the numbers your run produced)

| Manuscript item | Source file |
|---|---|
| Dataset description | `fig01_class_distribution`, `fig02_sample_gallery`, `table_dataset_split`, `table_leakage_audit` |
| Main comparison table | `table_main_results.tex / .csv` |
| Fusion-design ablation | `table_ablation.tex`, `fig12_ablation` |
| Statistical significance | `table_significance.csv`, `table_mcnemar_per_seed.csv` |
| Why fusion helps (complementarity) | `table_complementarity.csv`, `fig07_gate_analysis`, `table_gate_by_subset.csv` |
| Learning behaviour | `fig04_learning_curves` |
| Error analysis | `fig05_confusion_matrix_cgf`, `fig06_per_class_f1`, `table_top_confusions_cgf.csv` |
| Representation quality | `fig08_tsne_embeddings`, `table_silhouette.csv` |
| Calibration | `fig03_calibration` (+ ECE / NLL columns in the main table) |
| Robustness | `fig09_robustness`, `table_robustness_accuracy.csv` |
| Label efficiency | `fig11_label_efficiency`, `table_label_efficiency.csv` |
| Cost | `fig10_accuracy_vs_cost`, `table_efficiency.csv` |

**Figure placement.** Widths are 85 mm (single column, `COL1`) or 178 mm (double column, `COL2`). Insert the 600-dpi PNG (or the vector PDF) at 100 % scale. If your journal uses other widths, change `COL1` / `COL2` in Cell 2 and re-run the plotting cells only.

## Reporting guidance

* Report **mean ± SD over seeds**, and state that the test set was used once, after model selection on the validation set.
* Fusion gains on AID with 70 % training data are expected to be small in absolute terms; the label-efficiency, robustness and calibration sections are where architectural differences become visible. Report whichever direction the results take — including cases where a baseline is competitive.
* Late-fusion rows need *two* networks at inference; compare them to CGF-Net with the cost table in mind.
* `tta_*` columns are an add-on (8 dihedral views); the main comparison is single-view for every model.
* Robustness numbers come from a single seed's checkpoint; do not over-interpret sub-point differences.
* Limitations to state: a single dataset, one CNN and one ViT backbone, last-stage CNN features only (no multi-scale), fixed split.

## Crash recovery

Re-run the notebook from the top with the **same `RUN_ID` and settings**. Finished runs are skipped, an interrupted run resumes from its last completed epoch (`latest.pth`, written atomically), and a run that had already early-stopped is not trained further. A changed hyper-parameter raises an explicit error (config fingerprint) instead of silently corrupting the study — use a new `RUN_ID` for a new configuration.